In [11]:
## Imports and config
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
from pathlib import Path
import random
import re
import copy
import calendar


In [18]:

hf_token =json.load(open(Path("./config.json")))["hg_access_token"]
cache_dir =json.load(open(Path("./config.json")))["cache_dir"]
company_names_path=json.load(open(Path("./config.json")))["company_names_path"]
company_names= json.load(open(company_names_path))
time_frame={"daily",'weekly'}
months = {i: calendar.month_name[i] for i in range(1, 13)}


In [17]:
months

{'January': 1,
 'February': 2,
 'March': 3,
 'April': 4,
 'May': 5,
 'June': 6,
 'July': 7,
 'August': 8,
 'September': 9,
 'October': 10,
 'November': 11,
 'December': 12}

In [9]:
class QueryGenerator:
    def __init__(self, model, tokenizer, system_prompt, max_new_tokens=512, temperature=2.0):
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = system_prompt
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

    def build_messages(self, user_prompt):
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_prompt},
        ]

    def build_text(self, user_prompt):
        messages = self.build_messages(user_prompt)

        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            enable_thinking=False,
            add_generation_prompt=True,
        )

    def generate(self, user_prompt):
        text = self.build_text(user_prompt)

        model_inputs = self.tokenizer(
            [text],
            return_tensors="pt",
        ).to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            top_p= 0.8,
            top_k=20,
            min_p=0 
            
        )

        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

        model_output = self.tokenizer.decode(
            output_ids,
            skip_special_tokens=True,
        ).strip("\n")

        return model_output

In [ ]:
## Helper functions

def sample_spec(company_names, time_frame, months_dict, min_year=2006, max_year=2025):
    company_names_list = list(company_names.keys())
    time_frame_list = list(time_frame)
    
    companies = random.sample(company_names_list, k=random.randint(1, 3))
    time_frames = random.sample(time_frame_list, k=random.randint(1, 2))

    start_year = random.randint(min_year, max_year)
    end_year = random.randint(start_year, max_year)

    if start_year == end_year:
        # Sample two month keys (1-12) and sort them so month_1 <= month_2
        m1_key, m2_key = sorted(random.sample(list(months_dict.keys()), k=2))
    else:
        # If years are different, any 2 random months are fine
        m1_key, m2_key = random.sample(list(months_dict.keys()), k=2)

    # Convert keys to month names using the dictionary
    selected_months = [months_dict[m1_key], months_dict[m2_key]]

    return companies, time_frames, selected_months, start_year, end_year

def generate_user_pormpt(companies:list,features:list,months,start_year:int,end_year:int):
    return f"""Input specification:
    * Companies: {set(companies)}
    * Time Frames: {set(features)}
    * Start year: {months[0]} of {str(start_year)} 
    * End year: {months[1]} of {str(end_year)}
    Your output :
    """



def build_expected_output(companies, features, start_year, end_year):
    ### This output is aligned with Qwen3 native tool calling schema
    return {
    "action": "call",
    "function": "get_fundamentals",
    "arguments": {
        "queries": [
            {
                "symbols": [company],
                "metrics": [feature for feature in features],
                "start_year": start_year,
                "end_year": end_year,
            }
            for company in companies
            ]
                },
            }
       


def parse_model_output(model_output, expected_count=3):
    import re

    pattern = r"Q\d+:\s*(.*?)(?=\nQ\d+:|$)"
    queries = re.findall(pattern, model_output, flags=re.S)
    queries = [q.strip() for q in queries if q.strip()]

    if len(queries) != expected_count:
        return None

    return queries


def create_data_points(
    model_output,
    expected_output,
    companies,
    features,
    start_year,
    end_year,
    company_names,
    features_names,
    iteration=None,
    ):
    queries = parse_model_output(model_output, expected_count=2)

    if queries is None:
        failed_record = {
            "iteration": iteration,
            "raw_model_output": model_output,
            "expected_output": expected_output,
            "spec": {
                "companies": companies,
                "symbols": [company_names[company] for company in companies],
                "features": features,
                "metrics": [features_names[feature] for feature in features],
                "start_year": start_year,
                "end_year": end_year,
            },
        }

        return [], failed_record



    metadata = {
        "iteration": iteration,
        "companies": companies,
        "symbols": [company_names[company] for company in companies],
        "features": features,
        "metrics": [features_names[feature] for feature in features],
        "start_year": start_year,
        "end_year": end_year,
        "raw_model_output": model_output,
    }

    data_points = [
        {
            "query": query,
            "output_str": json.dumps(expected_output, ensure_ascii=False),
            "output_json": copy.deepcopy(expected_output),
            "metadata": copy.deepcopy(metadata),
        }
        for query in queries
    ]

    return data_points, None


def append_jsonl(path, records):
    with open(path, "a", encoding="utf-8") as fp:
        for record in records:
            fp.write(json.dumps(record, ensure_ascii=False) + "\n")


In [ ]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"
# model_name = "meta-llama/Llama-3.2-3B-Instruct"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    cache_dir=cache_dir,
    torch_dtype="auto",
    device_map="auto",
    
)


In [ ]:

# prepare the model input
SYSTEM_PROMPT_EASY = f"""You are an assistant of writing three diffrent querries regarding a dataset. This dataset consist of stock market companies list.
Each company has a CSV file with:
- Net income
- Instrinsic Value
- Revenue
- Diluted shares
as metrics, and mostly starts from 2009 to 2025. 

Your resposibilty is to create three queries for function calling. Your assistant will use your quesries and look for the corresponding stocks ticker and it features on specific CSV file.
The user gives you some input specification such as name of comapnies, Metrics (features) and period of time. You must generate the query based on this informattion  
In your query you must used vide range of vocabularies and use different grammers. Put yourself in postion of the user and use natural, common terms and words.
These three qurries should not be  similar to each other from grammer and vocabulary point of view.

You ourput should be like this:
Q1: 
Q2:
Q3:

Rules:
* Preserve every company, metric, and year exactly.
* Do not add or remove any requested information.
* You may vary the sentence structure and the order in which the year range, companies, and metrics appear.
* Do not use pronouns such as “it,” “the former,” or “the latter.”
* Do not ask for analysis, trends, growth, comparison, explanation, prediction, or recommendation.
"""




In [ ]:
all_data_points = []
failed_generations = []
query_generator= QueryGenerator(model,tokenizer,SYSTEM_PROMPT_EASY, max_new_tokens=512,temperature=0.7)


for i in range(700):
        if (i + 1) % 30 == 0:
            print("iteration:", i+1)
        companies, features, start_year,end_year= sample_spec(company_names, features_names)
        user_prompt= generate_user_pormpt(companies,features,start_year,end_year)
        print(user_prompt,end="\r")
        expected_output = build_expected_output(companies,features,start_year,end_year)
        
        model_output = query_generator.generate(user_prompt)
        
        data_points, failed_record = create_data_points(
        model_output=model_output,
        expected_output=expected_output,
        companies=companies,
        features=features,
        start_year=start_year,
        end_year=end_year,
        company_names=company_names,
        features_names=features_names,
        iteration=i,
    )

        if data_points:
            all_data_points.extend(data_points)
        else:
            failed_generations.append(failed_record)

        if (i + 1) % 100 == 0:
            append_jsonl(f"dataset_checkpoint_{i + 1}.jsonl", all_data_points)
            append_jsonl(f"failed_checkpoint_{i + 1}.jsonl", failed_generations)



append_jsonl(f"dataset_checkpoint.jsonl", all_data_points)
append_jsonl(f"failed_checkpoint.jsonl", failed_generations)